# Generate and validate DICOMDIR

Builds a DICOMDIR file for a Spectralis export by walking its `DICOM/` tree and hand-assembling the `DirectoryRecordSequence` (PATIENT -> STUDY -> SERIES -> IMAGE/RAW DATA/ENCAP DOC), instead of relying on `pydicom.fileset.FileSet` (which would rename/reorganize every file into its own `PTxxxxxx/STxxxxxx/SExxxxxx` layout) or DCMTK's `dcmmkdir` (which silently drops `OT`-modality Raw Data Storage instances).

Directory record offsets (`OffsetOfTheNextDirectoryRecord`, `OffsetOfReferencedLowerLevelDirectoryEntity`, and the two root-level offset elements) are computed with a two-pass write: write once with placeholder offsets, re-read to learn each record's real byte offset via `seq_item_tell`, patch the offsets, and write again.

The last section runs this over a list of batch folders: for each one, it regenerates a DICOMDIR from that folder's `DICOM/` files and compares it against the `DICOMDIR` already sitting in that same folder. A mismatch means the DICOM files no longer agree with the existing DICOMDIR (files added/removed/changed since it was last built) -- that batch needs reprocessing.

In [ ]:
import os

import pydicom
from pydicom.dataset import Dataset, FileDataset, FileMetaDataset
from pydicom.uid import ExplicitVRLittleEndian

# SOP Class UID for "Raw Data Storage" -- Spectralis stores its proprietary
# raw scan data (Modality OT) under this SOP class.
RAW_DATA_SOP_CLASS_UID = "1.2.840.10008.5.1.4.1.1.66"

## Build the DICOMDIR dataset skeleton

A bare `FileDataset` with the required File Meta elements. `MediaStorageSOPClassUID` must be the standard *Media Storage Directory Storage* UID (`1.2.840.10008.1.3.10`) -- readers such as `pydicom.fileset.FileSet` reject the file otherwise.

In [ ]:
def build_dicomdir_skeleton(output_file_path):
    file_meta = FileMetaDataset()
    file_meta.MediaStorageSOPClassUID = pydicom.uid.MediaStorageDirectoryStorage
    file_meta.MediaStorageSOPInstanceUID = pydicom.uid.generate_uid()
    file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
    file_meta.ImplementationClassUID = pydicom.uid.PYDICOM_IMPLEMENTATION_UID

    ds = FileDataset(output_file_path, {}, file_meta=file_meta, preamble=b"\x00" * 128)
    ds.is_little_endian = True
    ds.is_implicit_VR = False

    ds.FileSetID = "AIREADI"
    ds.FileSetConsistencyFlag = 0
    # Placeholders -- filled in once every record's real byte offset is known.
    ds.OffsetOfTheFirstDirectoryRecordOfTheRootDirectoryEntity = 0
    ds.OffsetOfTheLastDirectoryRecordOfTheRootDirectoryEntity = 0
    ds.DirectoryRecordSequence = []

    return ds

## Walk the DICOM tree and build the record hierarchy

For every file, ensure its PATIENT/STUDY/SERIES ancestor records exist (creating them the first time they're seen) and append an instance record. `children_of` tracks each record's ordered children (keyed by `id(record)`, with `None` standing for the root) so the offset chain can be computed afterwards.

In [ ]:
def add_directory_records(ds, root_dir, output_file_path):
    patient_records = {}
    study_records = {}
    series_records = {}
    children_of = {None: []}
    # Per-study counter for series whose source SeriesNumber is missing --
    # the original Spectralis-generated DICOMDIR numbers these 0, 1, 2, ...
    # in encounter order within a study rather than leaving them all at 0.
    auto_series_numbers = {}

    file_count = 0
    added_count = 0

    for root, _, files in os.walk(root_dir):
        for file in files:
            file_count += 1
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, os.path.dirname(output_file_path))
            rel_path_parts = rel_path.replace("/", "\\").split("\\")

            try:
                dcm = pydicom.dcmread(full_path, force=True)
                if "SOPClassUID" not in dcm or "SOPInstanceUID" not in dcm:
                    continue

                # Some Spectralis files carry fractional seconds (e.g. "142711.000000");
                # trim them so StudyTime matches the original export's format.
                if dcm.get("StudyTime"):
                    dcm.StudyTime = str(dcm.StudyTime).split(".")[0]

                # --- PATIENT level ---
                pid = getattr(dcm, "PatientID", "UNKNOWN_PATIENT")
                if pid not in patient_records:
                    pat_rec = Dataset()
                    pat_rec.OffsetOfTheNextDirectoryRecord = 0
                    pat_rec.RecordInUseFlag = 0xFFFF
                    pat_rec.OffsetOfReferencedLowerLevelDirectoryEntity = 0
                    pat_rec.DirectoryRecordType = "PATIENT"
                    pat_rec.PatientID = pid
                    pat_rec.PatientName = getattr(dcm, "PatientName", "UNKNOWN^PATIENT")
                    ds.DirectoryRecordSequence.append(pat_rec)
                    patient_records[pid] = pat_rec
                    children_of[id(pat_rec)] = []
                    children_of[None].append(pat_rec)
                pat_rec = patient_records[pid]

                # --- STUDY level ---
                study_uid = dcm.StudyInstanceUID
                if study_uid not in study_records:
                    study_rec = Dataset()
                    study_rec.OffsetOfTheNextDirectoryRecord = 0
                    study_rec.RecordInUseFlag = 0xFFFF
                    study_rec.OffsetOfReferencedLowerLevelDirectoryEntity = 0
                    study_rec.DirectoryRecordType = "STUDY"
                    study_rec.StudyDate = getattr(dcm, "StudyDate", "")
                    # StudyTime may be present but empty (None once read back) -- default
                    # to midnight instead of leaving it blank.
                    study_rec.StudyTime = getattr(dcm, "StudyTime", None) or "000000"
                    study_rec.StudyDescription = getattr(dcm, "StudyDescription", "")
                    study_rec.StudyInstanceUID = study_uid
                    ds.DirectoryRecordSequence.append(study_rec)
                    study_records[study_uid] = study_rec
                    children_of[id(study_rec)] = []
                    children_of[id(pat_rec)].append(study_rec)
                    auto_series_numbers[id(study_rec)] = 0
                study_rec = study_records[study_uid]

                # --- SERIES level ---
                series_uid = dcm.SeriesInstanceUID
                if series_uid not in series_records:
                    series_rec = Dataset()
                    series_rec.OffsetOfTheNextDirectoryRecord = 0
                    series_rec.RecordInUseFlag = 0xFFFF
                    series_rec.OffsetOfReferencedLowerLevelDirectoryEntity = 0
                    series_rec.DirectoryRecordType = "SERIES"
                    series_rec.Modality = getattr(dcm, "Modality", "OT")
                    series_rec.SeriesInstanceUID = series_uid
                    # SeriesNumber is Type 1 (required) but some Spectralis raw-data
                    # series carry the tag with an empty value, which pydicom reads
                    # back as None -- auto-number those sequentially within the study.
                    series_number = getattr(dcm, "SeriesNumber", None)
                    if series_number is None:
                        series_number = auto_series_numbers[id(study_rec)]
                        auto_series_numbers[id(study_rec)] += 1
                    series_rec.SeriesNumber = series_number
                    if "SeriesDescription" in dcm:
                        series_rec.SeriesDescription = dcm.SeriesDescription
                    ds.DirectoryRecordSequence.append(series_rec)
                    series_records[series_uid] = series_rec
                    children_of[id(series_rec)] = []
                    children_of[id(study_rec)].append(series_rec)
                series_rec = series_records[series_uid]

                # --- INSTANCE level ---
                inst_rec = Dataset()
                inst_rec.OffsetOfTheNextDirectoryRecord = 0
                inst_rec.RecordInUseFlag = 0xFFFF
                inst_rec.OffsetOfReferencedLowerLevelDirectoryEntity = 0

                if "EncapsulatedDocument" in dcm:
                    inst_rec.DirectoryRecordType = "ENCAP DOC"
                elif str(dcm.SOPClassUID) == RAW_DATA_SOP_CLASS_UID:
                    inst_rec.DirectoryRecordType = "RAW DATA"
                else:
                    inst_rec.DirectoryRecordType = "IMAGE"

                inst_rec.ReferencedFileID = rel_path_parts
                inst_rec.ReferencedSOPClassUIDInFile = dcm.SOPClassUID
                inst_rec.ReferencedSOPInstanceUIDInFile = dcm.SOPInstanceUID
                inst_rec.ReferencedTransferSyntaxUIDInFile = dcm.file_meta.TransferSyntaxUID
                inst_rec.InstanceNumber = getattr(dcm, "InstanceNumber", "1")

                ds.DirectoryRecordSequence.append(inst_rec)
                children_of[id(series_rec)].append(inst_rec)
                added_count += 1

            except Exception as e:
                print(f"Skipping damaged or unparseable file: {file}. Error: {e}")

    return children_of, file_count, added_count

## Order records and compute directory record offsets

Before writing, the root PATIENT list is sorted by `PatientID`, each PATIENT's STUDY children are sorted by `(StudyDate, StudyTime)`, and each STUDY's SERIES children by `SeriesNumber` (matching the order the original Spectralis-generated DICOMDIR uses), and `DirectoryRecordSequence` is rebuilt via a tree walk so the physical file order matches.

DICOMDIR readers (including `pydicom.fileset.FileSet`) walk the hierarchy purely via byte offsets, not the order of `DirectoryRecordSequence` -- each record needs to know the file offset of its next sibling and its first child, and the top-level dataset needs the offset of the first/last root record. Those offsets aren't known until the file has actually been encoded, so this writes twice: once with placeholders to learn every record's real offset (via `seq_item_tell`, which `dcmread` sets on any object read from a sequence), then again with the real offsets filled in. Because the offset elements are fixed-length (`UL`), patching their values doesn't change any byte lengths, so this second write is self-consistent.

In [ ]:
def sort_directory_records(ds, children_of):
    # Sort the root PATIENT list by PatientID, each PATIENT's STUDY children
    # by (StudyDate, StudyTime), and each STUDY's SERIES children by
    # (has SeriesDescription, SeriesNumber) -- the original Spectralis-
    # generated DICOMDIR groups all series without a SeriesDescription (e.g.
    # RAW DATA) before any series that has one, ordered by SeriesNumber
    # within each group, rather than interleaving by SeriesNumber alone.
    # Ties beyond that keep their original relative order (stable sort).
    # Also rebuilds DirectoryRecordSequence via a tree walk so the physical
    # file order matches.
    for siblings in children_of.values():
        if not siblings:
            continue
        record_type = siblings[0].DirectoryRecordType
        if record_type == "PATIENT":
            siblings.sort(key=lambda rec: rec.PatientID)
        elif record_type == "STUDY":
            siblings.sort(key=lambda rec: (rec.StudyDate, rec.StudyTime))
        elif record_type == "SERIES":
            siblings.sort(
                key=lambda rec: ("SeriesDescription" in rec, int(rec.SeriesNumber))
            )

    ordered = []

    def visit(parent_key):
        for child in children_of.get(parent_key, []):
            ordered.append(child)
            visit(id(child))

    visit(None)
    ds.DirectoryRecordSequence = ordered


def write_dicomdir_with_offsets(ds, children_of, output_file_path):
    sort_directory_records(ds, children_of)

    # Pass 1: write with placeholder (zero) offsets to learn each record's real byte offset.
    ds.save_as(output_file_path, enforce_file_format=True)

    reread = pydicom.dcmread(output_file_path)
    for original_rec, reread_rec in zip(ds.DirectoryRecordSequence, reread.DirectoryRecordSequence):
        original_rec._offset = reread_rec.seq_item_tell

    # Pass 2: fill in the real next-sibling / first-child offsets.
    for parent_key, siblings in children_of.items():
        for index, child in enumerate(siblings):
            child.OffsetOfTheNextDirectoryRecord = (
                siblings[index + 1]._offset if index + 1 < len(siblings) else 0
            )
        if parent_key is None and siblings:
            ds.OffsetOfTheFirstDirectoryRecordOfTheRootDirectoryEntity = siblings[0]._offset
            ds.OffsetOfTheLastDirectoryRecordOfTheRootDirectoryEntity = siblings[-1]._offset

    for record in ds.DirectoryRecordSequence:
        children = children_of.get(id(record), [])
        record.OffsetOfReferencedLowerLevelDirectoryEntity = children[0]._offset if children else 0

    ds.save_as(output_file_path, enforce_file_format=True)


def generate_dicomdir(dicom_dir, output_dicomdir_path):
    ds = build_dicomdir_skeleton(output_dicomdir_path)
    children_of, file_count, added_count = add_directory_records(ds, dicom_dir, output_dicomdir_path)
    write_dicomdir_with_offsets(ds, children_of, output_dicomdir_path)
    return file_count, added_count

## Compare two DICOMDIRs

Structural equality check instead of a text diff: for each DICOMDIR, build a `PatientID -> (StudyDate, StudyTime, StudyDescription) -> [series signatures]` index (one entry per distinct `SeriesInstanceUID`, so multi-instance series aren't double-counted), sort each series list, and compare. Sorting each level before comparing means differences in *order* (which record was walked/written first) don't count as mismatches -- only actual differences in which patients/studies/series exist do.

In [ ]:
from pydicom.fileset import FileSet


def series_signature(instance):
    return (
        instance.Modality,
        int(instance.SeriesNumber),
        getattr(instance, "SeriesDescription", ""),
        instance.node.record_type,
    )


def build_comparison_index(dicomdir_path):
    fs = FileSet(pydicom.dcmread(dicomdir_path))

    # One entry per SeriesInstanceUID, so a series with multiple instances
    # (e.g. multi-frame IMAGE series) isn't counted once per instance.
    series_by_uid = {}
    for instance in fs:
        # A couple of studies in the original DICOMDIR still carry
        # fractional-second StudyTime values (e.g. "115753.000000");
        # normalize on both sides so that doesn't register as a mismatch.
        study_time = str(instance.StudyTime).split(".")[0]
        series_by_uid[instance.SeriesInstanceUID] = (
            instance.PatientID,
            (instance.StudyDate, study_time, instance.StudyDescription),
            series_signature(instance),
        )

    index = {}
    for patient_id, study_key, series_key in series_by_uid.values():
        index.setdefault(patient_id, {}).setdefault(study_key, []).append(series_key)

    # Sort so list order doesn't affect the equality comparison below.
    for studies in index.values():
        for series_list in studies.values():
            series_list.sort(key=str)

    return index

In [ ]:
def diff_comparison_indexes(index_a, index_b):
    differences = []
    all_patients = sorted(set(index_a) | set(index_b))

    for patient_id in all_patients:
        studies_a = index_a.get(patient_id, {})
        studies_b = index_b.get(patient_id, {})
        all_studies = sorted(set(studies_a) | set(studies_b), key=str)

        for study_key in all_studies:
            series_a = studies_a.get(study_key, [])
            series_b = studies_b.get(study_key, [])
            if series_a != series_b:
                differences.append((patient_id, study_key, series_a, series_b))

    return differences

## Validate every batch folder under `batch_root`

`batch_root` is expected to contain one subfolder per batch, each with a `DICOM/` subfolder and an existing `DICOMDIR`. For every batch folder found directly under `batch_root`:

1. Regenerate a DICOMDIR from `DICOM/` into a sibling file named `DICOMDIR_generated`.
2. Compare it against the existing `DICOMDIR`.
3. If they match, delete `DICOMDIR_generated` -- nothing to do.
4. If they differ, rename the existing `DICOMDIR` to `DICOMDIR_old` (so the original is preserved) and rename `DICOMDIR_generated` to `DICOMDIR`, so the regenerated file is what gets uploaded to online storage for processing. The batch is recorded in `updated_batches`.

A batch is skipped (not touched) if it has no `DICOM/` subfolder or no existing `DICOMDIR`. If a `DICOMDIR_old` already exists for a batch (e.g. from a previous run), the batch is skipped with an error rather than silently overwriting it.</cell id="b1cba897">


In [ ]:
batch_root = r"D:\year3+raw\spectralis-s"

In [ ]:
batch_dirs = sorted(
    entry.path for entry in os.scandir(batch_root) if entry.is_dir()
)

updated_batches = []
skipped_batches = []
matched_batches = []
batch_differences = {}

for batch_dir in batch_dirs:
    print(f"Processing batch: {batch_dir}")
    dicom_dir = os.path.join(batch_dir, "DICOM")
    existing_dicomdir = os.path.join(batch_dir, "DICOMDIR")
    generated_dicomdir = os.path.join(batch_dir, "DICOMDIR_generated")
    old_dicomdir = os.path.join(batch_dir, "DICOMDIR_old")

    if not os.path.isdir(dicom_dir):
        print(f"SKIP      {batch_dir}  (no DICOM/ subfolder)")
        skipped_batches.append(batch_dir)
        continue
    if not os.path.isfile(existing_dicomdir):
        print(f"SKIP      {batch_dir}  (no existing DICOMDIR to compare against)")
        skipped_batches.append(batch_dir)
        continue
    if os.path.exists(old_dicomdir):
        print(f"SKIP      {batch_dir}  (DICOMDIR_old already exists -- investigate before rerunning)")
        skipped_batches.append(batch_dir)
        continue

    file_count, added_count = generate_dicomdir(dicom_dir, generated_dicomdir)
    existing_index = build_comparison_index(existing_dicomdir)
    generated_index = build_comparison_index(generated_dicomdir)
    differences = diff_comparison_indexes(existing_index, generated_index)

    if not differences:
        os.remove(generated_dicomdir)
        matched_batches.append(batch_dir)
        print(f"MATCH     {batch_dir}  ({added_count}/{file_count} files)")
    else:
        os.rename(existing_dicomdir, old_dicomdir)
        os.rename(generated_dicomdir, existing_dicomdir)
        updated_batches.append(batch_dir)
        batch_differences[batch_dir] = differences
        print(f"UPDATED   {batch_dir}  ({len(differences)} differences, {added_count}/{file_count} files)")

print(f"\n{len(matched_batches)} matched, {len(updated_batches)} updated, {len(skipped_batches)} skipped\n")
print("Batches updated (original preserved as DICOMDIR_old):")
for b in updated_batches:
    print(f"  {b}")

In [ ]:
if batch_differences:
    for batch_dir, differences in batch_differences.items():
        print(f"\n{batch_dir}:\n")
        for patient_id, study_key, existing_series, generated_series in differences:
            print(f"  PatientID={patient_id!r} {study_key}")
            print(f"    existing:  {existing_series}")
            print(f"    generated: {generated_series}")
else:
    print("No batches needed reprocessing.")